# Autoencoder and latent data

Train an autoencoder with Lightning, inspect its learned representation, then save a checkpoint and cache trajectories for [processor training](diffusion_and_flow_matching.ipynb). Use the [notebook environment](index.md); no earlier notebook or GPU is required.

In [ ]:
import lightning as L
import matplotlib.pyplot as plt
import torch
from _support import OUTPUT_ROOT
from omegaconf import OmegaConf

from autocast.data.encoded_dataset import CachedLatentDataset
from autocast.data.utils import get_autosim_datamodule
from autocast.decoders.dc import DCDecoder
from autocast.encoders.dc import DCEncoder
from autocast.models.autoencoder import AE
from autocast.scripts.cache_latents import cache_latents
from autocast.utils import get_optimizer_config
from autocast.utils.plots import plot_spatiotemporal_snapshots

torch.set_num_threads(2)
_ = L.seed_everything(42)
autoencoder_dir = OUTPUT_ROOT / "autoencoder"
latent_dir = autoencoder_dir / "cached_latents"

## Generate the data

As in the quickstart, AutoSim generates small advection–diffusion trajectories. `autoencoder_mode=True` makes each target identical to its input: reconstruction, not forecasting. Keep the loading options for the cache metadata later.

In [ ]:
data_options = {
    "data_path": str(autoencoder_dir / "data"),
    "n_steps_input": 1,
    "n_steps_output": 1,
    "stride": 1,
    "batch_size": 8,
    "num_workers": 0,
    "use_normalization": False,
}
datamodule = get_autosim_datamodule(
    "advection_diffusion",
    simulator_kwargs={"n": 16, "T": 1.0, "dt": 0.1},
    n_train=6,
    n_valid=2,
    n_test=2,
    autoencoder_mode=True,
    overwrite=True,
    seed=42,
    **data_options,
)

## Build and train

`DCEncoder` compresses each frame spatially; `DCDecoder` reconstructs it. `AE` supplies the reconstruction loss and Lightning training hooks. The constructor options are reused when loading the checkpoint.

In [ ]:
encoder_options = {
    "in_channels": 1,
    "out_channels": 2,
    "hid_channels": (8, 16),
    "hid_blocks": (1, 1),
    "pixel_shuffle": False,
}
decoder_options = {**encoder_options, "in_channels": 2, "out_channels": 1}
autoencoder = AE(
    encoder=DCEncoder(**encoder_options),
    decoder=DCDecoder(**decoder_options),
    optimizer_config=get_optimizer_config(learning_rate=3e-3),
)

In [ ]:
trainer = L.Trainer(
    max_epochs=10,
    accelerator="cpu",
    logger=False,
    enable_checkpointing=False,
    enable_progress_bar=False,
    enable_model_summary=False,
)
trainer.fit(autoencoder, datamodule=datamodule)

## Inspect reconstructions and latents

Encode the raw test trajectories frame by frame (normalization is disabled above). Change `sample_index` or `frame` in the plot cells without retraining.

In [ ]:
fields = datamodule.test_dataset.data
autoencoder.eval()
with torch.no_grad():
    latents = autoencoder.encoder.encode_tensor(fields)
    reconstruction = autoencoder.decode(latents)

fields.shape, latents.shape, reconstruction.shape

In [ ]:
sample_index = 0
figure = plot_spatiotemporal_snapshots(
    true=fields,
    pred=reconstruction,
    batch_idx=sample_index,
    timesteps=(0, 5, 10),
    pred_label="Reconstruction",
    title="Autoencoder reconstruction",
)
plt.show()

Time is preserved: each 16×16 field becomes an 8×8 grid with two latent channels. These are learned features, not physical fields.

In [ ]:
frame = 0
figure, axes = plt.subplots(1, latents.shape[-1], figsize=(6, 3))
for channel, ax in enumerate(axes):
    im = ax.imshow(latents[sample_index, frame, ..., channel], cmap="RdBu_r")
    ax.set_title(f"Latent channel {channel}")
    ax.set_axis_off()
    figure.colorbar(im, ax=ax, shrink=0.7)
plt.tight_layout()
plt.show()

## Save and reload

Provide matching encoder and decoder instances when loading the Lightning checkpoint.

In [ ]:
checkpoint = autoencoder_dir / "autoencoder.ckpt"
trainer.save_checkpoint(checkpoint)
reloaded = AE.load_from_checkpoint(
    checkpoint,
    encoder=DCEncoder(**encoder_options),
    decoder=DCDecoder(**decoder_options),
    map_location="cpu",
).eval()

with torch.no_grad():
    restored = reloaded.decode(reloaded.encoder.encode_tensor(fields))
torch.testing.assert_close(restored, reconstruction)

## Cache trajectories for reuse

The cache function takes configuration so later evaluation can reconstruct the model and locate the original data. Reuse the options above and save them beside the checkpoint. This is the same format used by the [CLI walkthrough](../walkthrough/autoencoder.md).

In [ ]:
cache_config = OmegaConf.create(
    {
        "datamodule": {
            "_target_": "autocast.data.datamodule.SpatioTemporalDataModule",
            **data_options,
            "autoencoder_mode": True,
        },
        "model": {
            "encoder": {
                "_target_": "autocast.encoders.dc.DCEncoder",
                **encoder_options,
            },
            "decoder": {
                "_target_": "autocast.decoders.dc.DCDecoder",
                **decoder_options,
            },
        },
        "autoencoder_checkpoint": str(checkpoint),
    }
)
OmegaConf.save(cache_config, autoencoder_dir / "resolved_autoencoder_config.yaml")
_ = cache_latents(cache_config, latent_dir, device="cpu")

The cache stores full trajectories, including their conditioning parameters. `CachedLatentDataset` selects windows at loading time; change the window lengths without encoding again.

In [ ]:
cached = CachedLatentDataset(
    latent_dir / "train",
    n_steps_input=2,
    n_steps_output=2,
    stride=2,
)
sample = cached[0]
sample.encoded_inputs.shape, sample.encoded_output_fields.shape

Continue with [diffusion and flow matching](diffusion_and_flow_matching.ipynb), which reuses this checkpoint and cache.